# 📊 Module 8: Numerical Stability Reference Guide

## 1️⃣ Core Numerical Issues — Quick Reference Table

| Issue | Definition | What Causes It | How to Detect | How to Fix |
|-------|-----------|----------------|---------------|------------|
| **Overflow** | Number too large for dtype to store | Repeated multiplication, exponentials, large factorials | Result = `inf`, `np.isinf()` | Use `float64`, log-space arithmetic, scaling |
| **Underflow** | Number too small, rounds to zero | Repeated division, tiny probabilities, exp(large negative) | Result = `0.0` when shouldn't be | Use `float64`, log-space arithmetic, rescaling |
| **Catastrophic Cancellation** | Subtracting similar large numbers loses precision | `large_num1 - large_num2` where values are close | Unexpected small/wrong results | Rearrange formula, use stable algorithms |
| **Precision Loss** | Accumulation of rounding errors | Summing many small values, float32 with large datasets | Wrong means/variances, failed tests | Use `float64`, Kahan summation, batch operations |

---

## 2️⃣ Float32 vs Float64 — Capacity & Precision

| dtype | Bytes | Range (approx) | Significant Digits | When to Use |
|-------|-------|----------------|-------------------|-------------|
| **float32** | 4 | ±3.4 × 10³⁸ | ~7 decimal digits | Large arrays where precision isn't critical, GPU computing |
| **float64** | 8 | ±1.8 × 10³⁰⁸ | ~15-16 decimal digits | Financial calculations, risk metrics, scientific computing (default) |

**Rule of thumb:** Use `float64` unless you have a specific reason (memory/speed) to use `float32`.

---

## 3️⃣ Overflow Deep Dive

### Definition
**Overflow** occurs when a calculation produces a number larger than the maximum representable value for the dtype.

### Common Causes in Quant Finance

| Scenario | Example | Why It Overflows |
|----------|---------|------------------|
| **Compound returns over long periods** | `np.prod(1 + returns)` for 10,000+ days | Multiplying 1.001 × 1.001 × ... (10,000 times) → huge number |
| **Exponential functions** | `np.exp(x)` where x > 700 | e^700 ≈ 10³⁰⁴ exceeds float64 max |
| **Factorial calculations** | `np.math.factorial(200)` | 200! is astronomically large |
| **Portfolio value simulation** | Daily compounding with high returns | Repeated multiplication without normalization |

### Detection
```python
result = np.prod(1 + returns)  # Might overflow

if np.isinf(result):
    print("Overflow detected!")
```

### Workarounds

#### **1. Log-Space Arithmetic**
```python
# WRONG (overflows)
cumulative = np.prod(1 + returns)

# RIGHT (stable)
log_cumulative = np.sum(np.log(1 + returns))
cumulative = np.exp(log_cumulative)
```

**Why it works:** Addition in log-space = multiplication in normal space, but stays within bounds.

#### **2. Rescaling/Normalization**
```python
# WRONG (overflows after many steps)
portfolio_value = 100
for r in returns:
    portfolio_value *= (1 + r)

# RIGHT (rescale periodically)
portfolio_value = 100
for i, r in enumerate(returns):
    portfolio_value *= (1 + r)
    if i % 252 == 0:  # Rescale annually
        portfolio_value = portfolio_value / 1e6  # Track in millions
```

#### **3. Use float64**
```python
# float32 overflows faster
returns_32 = returns.astype(np.float32)
cumulative_32 = np.cumprod(1 + returns_32)  # Might overflow

# float64 has larger range
cumulative_64 = np.cumprod(1 + returns)  # More resistant
```

---

## 4️⃣ Underflow Deep Dive

### Definition
**Underflow** occurs when a number is too small to represent and rounds to zero.

### Common Causes in Quant Finance

| Scenario | Example | Why It Underflows |
|----------|---------|-------------------|
| **Tiny probabilities** | Joint probability of 100 independent rare events | 0.01 × 0.01 × ... (100 times) → 10⁻²⁰⁰ |
| **Exponential decay** | `np.exp(-1000)` | e⁻¹⁰⁰⁰ ≈ 10⁻⁴³⁴, rounds to 0 |
| **Option pricing far OTM** | Black-Scholes for deep OTM options | Cumulative normal CDF returns near-zero |
| **Likelihood functions** | Product of many small densities | Each density < 1, product → 0 |

### Detection
```python
probabilities = np.array([0.001, 0.002, 0.0015, ...])  # 1000 values
joint_prob = np.prod(probabilities)

if joint_prob == 0.0:
    print("Underflow! True value is tiny but non-zero")
```

### Workarounds

#### **1. Log-Space Arithmetic**
```python
# WRONG (underflows)
joint_prob = np.prod(probabilities)  # → 0.0

# RIGHT (stable)
log_joint_prob = np.sum(np.log(probabilities))
# Work in log space, only exponentiate if needed
```

**For likelihood optimization:** Keep everything in log-space, maximize log-likelihood instead of likelihood.

#### **2. Rescaling**
```python
# WRONG (underflows in later iterations)
prob = 1.0
for p in tiny_probabilities:
    prob *= p

# RIGHT (rescale periodically)
prob = 1.0
scale_factor = 0
for p in tiny_probabilities:
    prob *= p
    if prob < 1e-100:  # Approaching underflow
        prob *= 1e50
        scale_factor -= 50  # Track the scaling
```

---

## 5️⃣ Catastrophic Cancellation

### Definition
**Catastrophic cancellation** occurs when subtracting two nearly equal large numbers, destroying significant digits.

### The Problem
```python
# Two large numbers that differ only in the last digits
a = 1.23456789012345  # 15 significant digits
b = 1.23456789012340  # Differs only in last digit

result = a - b  # = 0.00000000000005
# We've lost 13 digits of precision!
```

### Common Causes in Quant Finance

| Formula | Issue | Why It's Unstable |
|---------|-------|-------------------|
| **Variance (direct formula)** | `Var(X) = E[X²] - E[X]²` | When mean is large, both terms are large and close |
| **Quadratic formula** | `x = (-b ± sqrt(b² - 4ac)) / 2a` | When b² ≈ 4ac, subtraction loses precision |
| **Sharpe ratio differences** | `Sharpe_A - Sharpe_B` when both are similar | Direct subtraction magnifies errors |
| **Return differences** | `log(P₁/P₀) - log(P₂/P₀)` when prices close | Loss of precision in log differences |

### Example: Variance Calculation

```python
returns = np.random.randn(1000) * 0.001 + 1000  # Mean = 1000, std = 0.001

# METHOD A (UNSTABLE - catastrophic cancellation)
mean_squared = np.mean(returns**2)  # ≈ 1,000,000
squared_mean = np.mean(returns)**2  # ≈ 1,000,000
var_unstable = mean_squared - squared_mean  # Subtraction of ~equal numbers!
# Result: might be negative or wildly wrong

# METHOD B (STABLE - NumPy's built-in)
var_stable = np.var(returns)
# Uses: E[(X - μ)²] instead of E[X²] - E[X]²
```

**Why Method B is stable:** Subtracts mean BEFORE squaring, so you're squaring small deviations, not large values.

### Workarounds

#### **1. Rearrange the Formula**
```python
# UNSTABLE
variance = np.mean(x**2) - np.mean(x)**2

# STABLE (subtract mean first)
variance = np.mean((x - np.mean(x))**2)
```

#### **2. Use Stable Library Functions**
```python
# Don't implement variance yourself — use NumPy's stable version
variance = np.var(x)  # Uses Welford's algorithm internally
```

#### **3. Center Data First**
```python
# If you must use direct formula, center the data
x_centered = x - np.mean(x)
variance = np.mean(x_centered**2)  # Now mean is zero, no cancellation
```

---

## 6️⃣ Precision Loss from Accumulation

### Definition
Rounding errors compound when performing many operations, especially with `float32`.

### Common Causes

| Scenario | Why It Happens | Impact |
|----------|----------------|--------|
| **Averaging millions of values** | Each value rounded, errors accumulate in sum | Mean is slightly off |
| **Monte Carlo with float32** | 1M paths × small payoffs, rounding in each sum | Option price wrong by cents |
| **Cumulative sums over long series** | Each addition has rounding error | Final sum drifts from true value |
| **Iterative algorithms** | Errors compound over iterations | Optimization converges to wrong value |

### Example: Monte Carlo Precision Loss

```python
# Simulate option payoffs (many small values)
payoffs_64 = np.random.randn(1_000_000) * 0.0001  # float64 default
payoffs_32 = payoffs_64.astype(np.float32)

mean_64 = np.mean(payoffs_64)  # High precision
mean_32 = np.mean(payoffs_32)  # Precision loss in summation

print(f"Difference: {abs(mean_64 - mean_32):.2e}")  # Might be 1e-7
# For a $100 option, that's $0.00001 error — acceptable
# For a $10,000 portfolio, that's $0.001 error — might matter
```

### Workarounds

#### **1. Use float64**
```python
# Always use float64 for financial calculations
data = np.array([...], dtype=np.float64)
```

#### **2. Kahan Summation (Compensated Summation)**
```python
def kahan_sum(arr):
    """More accurate summation that compensates for rounding errors"""
    total = 0.0
    c = 0.0  # Compensation for lost low-order bits
    for value in arr:
        y = value - c
        t = total + y
        c = (t - total) - y  # Recover lost bits
        total = t
    return total
```

**When to use:** Summing millions of small values where precision matters.

#### **3. Batch Operations**
```python
# LESS PRECISE (long sequential sum)
total = 0
for value in million_values:
    total += value

# MORE PRECISE (NumPy's optimized sum)
total = np.sum(million_values)  # Uses pairwise summation internally
```

---

## 7️⃣ Special Values: NaN and Inf

### NaN (Not a Number)

| Cause | Example | How to Check | How to Handle |
|-------|---------|--------------|---------------|
| **Invalid operation** | `np.sqrt(-1)`, `0/0`, `inf - inf` | `np.isnan(x)` | Replace with default, drop, or use `np.nan_to_num()` |
| **Missing data propagation** | Any operation with NaN | `np.any(np.isnan(arr))` | `np.nanmean()`, `np.nanstd()` ignore NaNs |
| **Log of zero/negative** | `np.log(0)`, `np.log(-5)` | Check before operation | Add small epsilon: `np.log(x + 1e-10)` |

```python
# Detecting NaN
data = np.array([1, 2, np.nan, 4])
has_nan = np.any(np.isnan(data))  # True

# Handling NaN
mean_skip = np.nanmean(data)  # Computes mean ignoring NaN
data_clean = data[~np.isnan(data)]  # Filter out NaNs
data_filled = np.nan_to_num(data, nan=0.0)  # Replace NaN with 0
```

### Inf (Infinity)

| Cause | Example | How to Check | How to Handle |
|-------|---------|--------------|---------------|
| **Overflow** | `np.exp(1000)`, large products | `np.isinf(x)` | Use log-space, clip values, or rescale |
| **Division by zero** | `1/0` (with warnings off) | `np.isinf(x)` | Check denominator, add small epsilon |
| **Very large exponents** | `10**500` | `np.isinf(x)` | Use logarithms |

```python
# Detecting Inf
result = np.exp(1000)
is_infinite = np.isinf(result)  # True

# Preventing Inf in division
denominator = np.array([1, 2, 0, 4])
safe_result = np.divide(1, denominator, 
                        out=np.zeros_like(denominator, dtype=float),
                        where=denominator!=0)
# Or: denominator + 1e-10 to avoid zero
```

### NaN vs Inf Comparison

| Property | NaN | Inf |
|----------|-----|-----|
| **Equality** | `np.nan != np.nan` (True!) | `np.inf == np.inf` (True) |
| **Comparison** | Always returns False | Can compare: `x < np.inf` |
| **Propagation** | Any operation with NaN → NaN | `inf + 1 = inf`, `inf * 2 = inf` |
| **Boolean context** | `bool(np.nan)` → True (surprising!) | `bool(np.inf)` → True |

---

## 8️⃣ Practical Checklist — When to Worry

### ✅ Always Use Stable Approaches For:

| Task | Unstable Method | Stable Method |
|------|-----------------|---------------|
| **Variance** | `mean(x²) - mean(x)²` | `np.var(x)` or `mean((x - mean(x))²)` |
| **Cumulative returns (long series)** | `np.prod(1 + returns)` | `np.exp(np.sum(np.log(1 + returns)))` |
| **Probabilities (tiny values)** | `np.prod(probabilities)` | `np.sum(np.log(probabilities))` |
| **Standard deviation** | `sqrt(mean(x²) - mean(x)²)` | `np.std(x)` |
| **Correlation with large means** | Manual covariance/variance | `np.corrcoef(x, y)` (centers data internally) |

### 🚨 Red Flags in Code Review

```python
# 🚨 RED FLAG: Manual variance with large numbers
variance = np.mean(data**2) - np.mean(data)**2

# 🚨 RED FLAG: Long products without log-space
cumulative = np.prod(daily_multipliers)  # 1000+ elements

# 🚨 RED FLAG: float32 for financial calculations
prices = prices.astype(np.float32)  # Precision loss likely

# 🚨 RED FLAG: Division without zero-check
ratio = numerator / denominator  # What if denominator = 0?

# 🚨 RED FLAG: Exponential without overflow check
result = np.exp(large_value)  # Could be inf
```

### ✅ Green Flags (Good Practices)

```python
# ✅ Use built-in stable functions
variance = np.var(data)

# ✅ Log-space for long products
log_cumulative = np.sum(np.log(1 + returns))
cumulative = np.exp(log_cumulative)

# ✅ float64 by default (NumPy's default)
data = np.array([...])  # Automatically float64

# ✅ Safe division with where clause
result = np.divide(num, denom, where=denom!=0, out=np.zeros_like(num))

# ✅ Check for special values
if np.any(np.isnan(result)) or np.any(np.isinf(result)):
    raise ValueError("Numerical instability detected")
```

---

## 9️⃣ Quick Decision Tree

```
Do I need to multiply many numbers together?
├─ YES, <100 values
│  └─ Use np.prod() (usually fine)
├─ YES, >100 values or long time series
│  └─ Use log-space: np.exp(np.sum(np.log(values)))
└─ NO
   └─ Continue...

Do I need to calculate variance/covariance?
├─ YES
│  ├─ Use np.var(), np.cov(), np.corrcoef()
│  └─ NEVER use mean(x²) - mean(x)² manually
└─ NO
   └─ Continue...

Am I summing millions of small values?
├─ YES
│  ├─ Use float64
│  └─ Use np.sum() (has built-in compensation)
└─ NO
   └─ Continue...

Am I exponentiating or taking logs?
├─ YES
│  ├─ Check domain (log requires x > 0)
│  ├─ Check range (exp can overflow if x > 700)
│  └─ Add epsilon if needed: log(x + 1e-10)
└─ NO
   └─ You're probably safe!
```

---

## 🔟 Real-World Example: Portfolio Risk Calculation

```python
def calculate_portfolio_var_UNSTABLE(returns, weights):
    """DON'T DO THIS - Multiple numerical stability issues"""
    # Issue 1: Manual variance (catastrophic cancellation risk)
    mean_returns = np.mean(returns, axis=0)
    variance = np.mean(returns**2, axis=0) - mean_returns**2  # 🚨
    
    # Issue 2: float32 for faster computation (precision loss)
    weights = weights.astype(np.float32)  # 🚨
    
    # Issue 3: No checks for NaN/Inf
    portfolio_var = weights @ np.outer(variance, variance) @ weights  # 🚨
    
    return portfolio_var


def calculate_portfolio_var_STABLE(returns, weights):
    """Stable implementation"""
    # Use NumPy's stable covariance
    cov_matrix = np.cov(returns, rowvar=False)  # ✅
    
    # Keep float64 precision
    weights = np.asarray(weights, dtype=np.float64)  # ✅
    
    # Calculate variance
    portfolio_var = weights @ cov_matrix @ weights  # ✅
    
    # Check for numerical issues
    if np.isnan(portfolio_var) or np.isinf(portfolio_var):
        raise ValueError("Numerical instability in variance calculation")  # ✅
    
    return portfolio_var
```

---

**Save this as your Module 8 reference!** 🎯